# Day 01 — PyTorch 分布式训练基础：集合通信与 DDP

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Papa-Panda/field-notes/blob/main/pytorch/day-01-distributed-ddp.ipynb)

- **日期**：2026-09-30
- **仓库**：`Papa-Panda/field-notes`（private，个人训练场）
- **定位**：6–12 个月转型计划里「补齐 PyTorch Distributed 短板」的第一块砖；同时对齐 Inception 面试的两个实操轮次 —— **Research pytorch**（调试手感）和 **RL infra foundations**（并行概念 + coding）。
- **运行环境**：本 notebook 全部代码用 CPU + `gloo` 后端，Colab 免费版可直接跑。GPU/NCCL 版本标注为「待 H100」。


## 今日目标

1. 能从第一性原理写出 **ring all-reduce** 的通信量与时间公式（符号、量级都要对）。
2. 理解 **DDP 的 gradient bucketing + 通信/计算重叠** 机制，能解释"为什么 DDP 扩展效率高"。
3. 跑通 **2 进程 DDP 最小闭环**，并用 `all_gather` 验证梯度跨 rank 同步正确。
4. 整理一份 **Research pytorch 轮的 DDP 调试清单**（面试实操向）。


## 0. 为什么从这里开始（context 映射）

career thread（2026-09-30 三方校准）的结论：

- Inception 五个轮次单看都不特殊，**特殊的是组合** —— 整个 loop 全押 implementation/infra。Research pytorch 和 parallelism 恰好是投入产出比最高的两项。
- 缺口清单里 system design / ML-RL 理论 / general coding / behavioral 另起炉灶补，**不挤占实操主线**。
- 45 天 Foundation 计划里 PyTorch Distributed（DDP/FSDP）本来就是第一阶段；之前在 `post-training/ai-infra/r2-day-02` 跑过一个 CPU proxy（2-rank gloo，通信占比 ~35%），今天把原理和手感一次做扎实。

记号约定：$N$ = world size（rank 总数），$S$ = tensor 大小（bytes），$B$ = 单链路带宽（bytes/s），$\alpha$ = 单次点对点延迟（s）。


## 1. 集合通信原语（collectives）

| 原语 | 语义 | 在 DDP / 训练里用在哪 |
|---|---|---|
| `broadcast` | rank 0 → 全员分发同一份 tensor | 参数初始化同步、ckpt 分发 |
| `all_reduce` | 全员各自贡献一份，规约（sum）后人手一份结果 | **DDP 梯度同步**（sum 后除以 $N$） |
| `all_gather` | 全员各自贡献一份，拼起来后人手一份完整拼接 | ZeRO-3 参数 gather、验证一致性 |
| `reduce_scatter` | 全员各自贡献一份，规约后按 rank 切分分发 | ZeRO-1/2 梯度分片规约 |

核心直觉：**all-reduce = 每个 rank 既是数据的生产者也是消费者**，没有中心瓶颈 —— 这是它相对 parameter server 的结构性优势。


## 2. Ring all-reduce 的数学（第一性原理）

把大小为 $S$ 的 tensor 切成 $N$ 个 chunk（每 chunk $S/N$），$N$ 个 rank 排成环。

- **Step 1 — scatter-reduce**：$N-1$ 轮，每轮每个 rank 把自己手里的一个 chunk 发给右邻、从左邻收一个并累加。$N-1$ 轮后每个 rank 恰好拥有一个**完全规约好**的 chunk。
- **Step 2 — all-gather**：再 $N-1$ 轮，把规约好的 chunk 在环上传一圈，每人集齐 $N$ 个 chunk。

每轮每 rank 发送 $S/N$ bytes，共 $2(N-1)$ 轮：

$$\text{每 rank 总发送量} = 2(N-1) \cdot \frac{S}{N} = \frac{2(N-1)}{N}S$$

$$t_{\text{allreduce}} \approx 2(N-1)\alpha + \frac{2(N-1)}{N}\cdot\frac{S}{B}$$

两个关键推论（面试常考点）：

1. **带宽最优**：$N \to \infty$ 时每 rank 发送量 $\to 2S$，与 $N$ 无关。对比 parameter server：server 要收发 $2S \cdot N$（随 $N$ 线性涨，server 网卡是单点瓶颈）。
2. **延迟项随 $N$ 线性涨**：小 tensor（$S$ 小）时 $2(N-1)\alpha$ 主导 —— 这就是 DDP 要做 **gradient bucketing**（把小梯度拼成大 bucket 再发一次）的根本原因：把 $k$ 次小 all-reduce 合并成 1 次，延迟项从 $k \cdot 2(N-1)\alpha$ 降到 $2(N-1)\alpha$。


In [ ]:
def ring_allreduce_time(S_GB, N, B_GBs, alpha_us=20):
    """ring all-reduce 时间估计（ms）。S: GB, B: GB/s, alpha: us"""
    S, B = S_GB * 1e9, B_GBs * 1e9
    t = 2 * (N - 1) * alpha_us * 1e-6 + 2 * (N - 1) / N * S / B
    return t * 1e3

print("1GB tensor 的 all-reduce 时间估计：")
for N, B, link in [(8, 50, "NVLink-ish"), (64, 25, "IB 200Gb/s"), (1024, 25, "IB 200Gb/s")]:
    t = ring_allreduce_time(1.0, N, B)
    bw_term = 2 * (N - 1) / N * 1.0 / B * 1e3
    print(f"  N={N:4d} [{link:12s}] total={t:6.1f} ms  (带宽项 {bw_term:.1f} ms, 其余为延迟项)")


## 3. DDP 机制：为什么扩展效率高

**数据流**（每个 step）：

1. 每个 rank 持有**完整模型副本** + **1/N 的数据分片**，各自 forward + backward。
2. backward 过程中，梯度按 bucket 做 **all-reduce（sum）再除以 $N$** —— 数学上等价于在大 batch 上算梯度。
3. 参数更新：各 rank 用**相同**的平均梯度做 SGD/Adam，参数永远保持一致（要求初始参数一致 → 见调试清单第 2 条）。

**两个关键工程优化**：

- **Gradient bucketing**：按参数在模型中的**逆序**（backward 产生梯度的顺序）把梯度拼进 bucket（默认 25MB）。bucket 一满就**立即**发起 all-reduce，不必等整个 backward 结束。
- **通信/计算重叠**：前一层参数的 all-reduce 与后一层参数的 backward 计算**同时**进行。理想情况下单步 wall time $\approx \max(t_{compute}, t_{comm})$ 而非两者相加。

单步通信量（$P$ = 参数 bytes，fp32 下 $P = 4 \times$ #params）：

$$t_{comm} \approx \frac{2(N-1)}{N}\cdot\frac{P}{B}$$

> 数量级体感：70B 参数 fp32 → $P=280$GB。$N=1024$、$B=25$GB/s 时 $t_{comm} \approx 2 \times 280/25 \approx 22$s/step —— 这就是为什么大模型必须上 ZeRO/FSDP（分片 $P$）+ TP/PP（切模型），纯 DDP 撑不住。Day 02 讲。


In [ ]:
import os, torch
import torch.distributed as dist
import torch.multiprocessing as mp
from torch.nn.parallel import DistributedDataParallel as DDP

def run(rank, world_size):
    os.environ['MASTER_ADDR'] = '127.0.0.1'
    os.environ['MASTER_PORT'] = '29501'
    dist.init_process_group('gloo', rank=rank, world_size=world_size)

    torch.manual_seed(0)  # DDP 铁律：各 rank 初始参数必须一致
    model = DDP(torch.nn.Linear(10, 1))
    opt = torch.optim.SGD(model.parameters(), lr=0.1)

    x = torch.randn(32, 10)
    y = torch.randn(32, 1)
    opt.zero_grad()
    loss = ((model(x) - y) ** 2).mean()
    loss.backward()   # ← DDP 在这里 hook 了 gradient bucketing + all-reduce
    opt.step()

    # 验证：all_gather 各 rank 更新后的参数，应 bit 级一致
    params = torch.cat([p.data.view(-1) for p in model.parameters()])
    gathered = [torch.zeros_like(params) for _ in range(world_size)]
    dist.all_gather(gathered, params)
    if rank == 0:
        for r in range(1, world_size):
            d = (gathered[0] - gathered[r]).abs().max().item()
            print(f'rank0 vs rank{r}: 参数最大差 = {d:.2e}  (应为 0)')
        print(f'loss = {loss.item():.4f}')
    dist.destroy_process_group()

if __name__ == '__main__':
    mp.spawn(run, args=(2,), nprocs=2, join=True)


## 4. 实验：拟合 $\alpha$–$\beta$ 通信模型（2 rank, gloo, CPU）

用实测拟合 $t(S) = a + b\cdot S$，其中 $a = 2(N-1)\alpha$、$b = 2(N-1)/(N\cdot B)$，反解出延迟 $\alpha$ 和有效带宽 $B$。
这是在 infra 面试里**现场估算**通信开销的手感来源（"1024 张卡 all-reduce 1GB 要多久"这类题）。


In [ ]:
import os, time, torch
import torch.distributed as dist
import torch.multiprocessing as mp

def bench(rank, world_size):
    os.environ['MASTER_ADDR'] = '127.0.0.1'
    os.environ['MASTER_PORT'] = '29502'
    dist.init_process_group('gloo', rank=rank, world_size=world_size)
    N = world_size
    sizes = [4*1024, 64*1024, 1024*1024, 16*1024*1024]  # 4KB ~ 16MB
    if rank == 0:
        print(f'{"size":>10s} {"t/op (ms)":>12s}')
    ts = []
    for S in sizes:
        t = torch.zeros(S // 4)  # float32
        for _ in range(5):  # warmup
            dist.all_reduce(t)
        dist.barrier()
        t0 = time.perf_counter()
        iters = 20
        for _ in range(iters):
            dist.all_reduce(t)
        dt = (time.perf_counter() - t0) / iters * 1e3
        ts.append(dt)
        if rank == 0:
            print(f'{S//1024:>7d} KB {dt:>12.3f}')
    if rank == 0:
        # 最小二乘拟合 t = a + b*S
        import numpy as np
        S_arr = np.array(sizes, dtype=float)
        t_arr = np.array(ts)
        b, a = np.polyfit(S_arr, t_arr, 1)  # t(ms) = a + b*S
        alpha_us = a / (2 * (N - 1)) * 1e3
        B_GBs = 2 * (N - 1) / N / (b * 1e-3) / 1e9
        print(f'\n拟合: t = {a:.3f} ms + {b*1e6:.4f} us/KB')
        print(f'反解: alpha ≈ {alpha_us:.1f} us/op, 有效带宽 B ≈ {B_GBs:.2f} GB/s')
        print('（gloo/TCP 的带宽远低于 NVLink/IB，数量级对即可；换 NCCL+GPU 重测）')
    dist.destroy_process_group()

if __name__ == '__main__':
    mp.spawn(bench, args=(2,), nprocs=2, join=True)


## 5. Research pytorch 轮：DDP 调试清单（面试实操向）

1. **Hang 住不动**：最常见。某 rank 没进入 collective（分支逻辑导致调用次数不一致）或 rendezvous 失败。先看 `NCCL_DEBUG=INFO` 日志卡在哪一次 collective；`TORCH_DISTRIBUTED_DEBUG=DETAIL` 看 bucket 状态。
2. **梯度/参数跨 rank 不一致**：`torch.manual_seed` 没同步（模型 init、dataloader shuffle、`DistributedSampler` 漏配）。DDP 要求**初始参数 bit 级一致**，否则 all-reduce 平均的是错的东西。
3. **loss 不降、单卡却正常**：检查 `BatchNorm` —— 统计量没跨卡同步。用 `SyncBatchNorm` 或换 `LayerNorm`。
4. **`find_unused_parameters=True` 的代价**：为找 unused 参数多做一次全图遍历 + 额外 all-reduce。只有**真的**有参数不参与 forward 时才开；能静态确认就别开。
5. **混合精度顺序**：`GradScaler` 包在 DDP 外层（scaler.step → scaler.update），unscale 在 all-reduce 之后做。顺序错了会 silently 发散。
6. **torchrun vs mp.spawn**：`torchrun --nproc_per_node=2 --rdzv_backend=c10d` 是生产标准写法（elastic、自动 rendezvous）；本 notebook 用 `mp.spawn` 是为了单文件可跑。同文件已存为 `day-01-ddp-minimal.py`，可用 `torchrun` 直接起。
7. **NCCL 环境变量三件套**：`NCCL_DEBUG=INFO`（定位 hang）、`NCCL_IB_DISABLE=1`（无 IB 时强制走 TCP/socket 排查）、`TORCH_NCCL_ASYNC_ERROR_HANDLING=1`。

> Colab 里 `mp.spawn` 若报 `__main__` 相关错误：把 `day-01-ddp-minimal.py` 下载下来用 `!torchrun --nproc_per_node=2 day-01-ddp-minimal.py` 跑，这是标准姿势。


## 6. 小结

- **公式**：ring all-reduce 每 rank 发送 $\frac{2(N-1)}{N}S$，时间 $t \approx 2(N-1)\alpha + \frac{2(N-1)}{N}\frac{S}{B}$。$N$ 大时带宽项趋近 $2S/B$（与 $N$ 无关），延迟项随 $N$ 线性涨。
- **机制**：DDP = 数据并行 + gradient bucketing + 通信/计算重叠；梯度 all-reduce 平均后各 rank 参数天然一致。
- **手感**：2 进程 gloo 最小闭环跑通；$\alpha$–$\beta$ 拟合是以后估算任何并行方案通信开销的通用手法。
- **面试映射**：Research pytorch（DDP 调试清单 √）、RL infra foundations（并行概念 √，TP coding 待 Day 03）。

**Day 02 预告 — ZeRO / FSDP：打碎 $P$ 的内存墙**
Adam 训 $P$ 参数（fp32）要 $2P$（参数+梯度，fp16/bf16 下）$+ 4P + 4P$（动量+方差 fp32）$+ 4P$（fp32 主权重）$\approx 14$–$16P$ bytes。70B 模型 ≈ 1TB+，单卡放不下。ZeRO-1/2/3 把优化器状态、梯度、参数按 $N$ 切分；FSDP 是 PyTorch 原生实现。Day 02 推一遍 ZeRO 的通信量公式（ZeRO-3 每 step 多 1.5× 通信），并跑通 FSDP 最小例子（CPU 可跑，慢）。
